In [2]:
!pip install -q transformers evaluate datasets torch

In [3]:
from datasets import load_dataset
data = load_dataset('allenai/sciq')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
data

DatasetDict({
    train: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 11679
    })
    validation: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
})

In [5]:
data['train'].shape

(11679, 6)

In [6]:
data['train'][0]

{'question': 'What type of organism is commonly used in preparation of foods such as cheese and yogurt?',
 'distractor3': 'viruses',
 'distractor1': 'protozoa',
 'distractor2': 'gymnosperms',
 'correct_answer': 'mesophilic organisms',
 'support': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.'}

In [7]:
import random
from transformers import set_seed

set_seed(42)
rng = random.Random(42)

In [8]:

def process(example):
    question = example['question']
    context =  example['support']
    choices = [
        example['correct_answer'],
        example['distractor1'],
        example['distractor2'],
        example['distractor3']
    ]
    correct = example['correct_answer']
    rng.shuffle(choices)
    # shuffles the options to get a random answer other wise model will look at the statistical posi
    label = choices.index(correct)
    return {
        "question": example["question"],
        "support": example["support"],
        "choices": choices,
        "label": label
    }


In [9]:
dataset = data.map(process)

Map:   0%|          | 0/11679 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
dataset['train'][0]

{'question': 'What type of organism is commonly used in preparation of foods such as cheese and yogurt?',
 'distractor3': 'viruses',
 'distractor1': 'protozoa',
 'distractor2': 'gymnosperms',
 'correct_answer': 'mesophilic organisms',
 'support': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.',
 'choices': ['gymnosperms', 'protozoa', 'viruses', 'mesophilic organisms'],
 'label': 3}

In [11]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    'roberta-base'
)

In [12]:
def preprocess_function(examples):

    first_sentences = []
    second_sentences = []

    for q, support, choices in zip(
        examples["question"],
        examples["support"],
        examples["choices"]
    ):

        first_sentences.extend(
            [q] * 4
        )

        second_sentences.extend(
            [
                support + " " + tokenizer.sep_token + " " + choice
                for choice in choices
            ]
        )

    tokenized = tokenizer(
    first_sentences,
    second_sentences,
    truncation=True,
    max_length=256
    )

    result = {
        k: [
            v[i:i+4]
            for i in range(0, len(v), 4)
        ]
        for k, v in tokenized.items()
    }

    result["labels"] = examples["label"]

    return result

In [13]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/11679 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [14]:
sample = tokenized_dataset["train"][0]

print(sample.keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [15]:
len(sample["input_ids"])

4

In [16]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained(
    'roberta-base'
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | MISSING    | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 
roberta.pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
sample = tokenized_dataset["train"][0]

In [20]:
from dataclasses import dataclass
from typing import Optional, Union
import torch

@dataclass
class dataCollator:

    tokenizer: object
    padding: Union[bool, str] = True
    max_length: Optional[int] = None

    def __call__(self, features):

        labels = [feature.pop("labels") for feature in features]

        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened_features = []

        for feature in features:
            for i in range(num_choices):
                flattened_features.append({
                    k: v[i]
                    for k, v in feature.items()
                })

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt"
        )

        batch = {
            k: v.view(batch_size, num_choices, -1)
            for k, v in batch.items()
        }

        batch["labels"] = torch.tensor(labels)

        return batch

In [21]:
data_collator = dataCollator(
    tokenizer=tokenizer
)

In [22]:
batch = data_collator(
    [tokenized_dataset["train"][0],
     tokenized_dataset["train"][1]]
)

for k,v in batch.items():
    print(k, v.shape)

input_ids torch.Size([2, 4, 119])
attention_mask torch.Size([2, 4, 119])
labels torch.Size([2])


In [23]:
!pip install -q evaluate

In [24]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [25]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./roberta-sciq",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=4,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=50,

    load_best_model_at_end=True,

    fp16=False,
    bf16=False,

    report_to="none"
)

In [26]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

sample = data_collator([
    tokenized_dataset["train"][0],
    tokenized_dataset["train"][1]
])

sample = {k: v.to(device) for k, v in sample.items()}

outputs = model(**sample)

print(outputs.logits)
print(outputs.loss)

tensor([[0.1646, 0.1666, 0.1667, 0.1671],
        [0.1835, 0.1855, 0.1824, 0.1823]], device='cuda:0',
       grad_fn=<ViewBackward0>)
tensor(1.3864, device='cuda:0', grad_fn=<NllLossBackward0>)


In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.287273,0.341595,0.868000
2,0.313783,0.362535,0.870000
3,0.188220,0.477800,0.872000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.287273,0.341595,0.868000
2,0.313783,0.362535,0.870000
3,0.188220,0.477800,0.872000
4,0.210914,0.529938,0.871000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5840, training_loss=0.29707811074714136, metrics={'train_runtime': 7905.8239, 'train_samples_per_second': 5.909, 'train_steps_per_second': 0.739, 'total_flos': 2.0624136674630784e+16, 'train_loss': 0.29707811074714136, 'epoch': 4.0})

In [29]:
trainer.save_model("./roberta_sciq_model")
tokenizer.save_pretrained("./roberta_sciq_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./roberta_sciq_model/tokenizer_config.json',
 './roberta_sciq_model/tokenizer.json')